# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kishiagaytano/wilt/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane 4: CTR / Engagement Opportunity Scoring.** I am scoring which visible pages under-capture clicks relative to what their search position should earn, and ranking them so an editor reviews the highest-value ones first.

I checked three other lanes against the data rather than their descriptions. Lane 4 was the one with a large usable population, a real measured relationship, and a gap a flat rule demonstrably cannot rank.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/kishiagaytano/wilt"
REPO_DIR = "wilt"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

TIER_ORDER = ["top_3", "page_1", "striking", "page_3_5", "deep"]

# Does a usable population survive a volume floor? (the floor itself is chosen in ML-04/ML-07)
print(f"{'floor':>7}{'pages':>9}{'clients':>9}{'med clicks':>12}{'<=5 clicks':>12}")
for f in (500, 1000, 2000, 5000, 10000):
    s = df[(df["impressions_90d"] >= f) & (df["avg_position"] > 0)]
    print(f"{f:>7,}{len(s):>9,}{s['client_id'].nunique():>9}"
          f"{s['clicks_90d'].median():>12,.0f}{(s['clicks_90d'] <= 5).mean():>11.1%}")

  floor    pages  clients  med clicks  <=5 clicks
    500   16,726       28           5      51.3%
  1,000   13,512       28           8      40.9%
  2,000   10,215       28          14      27.2%
  5,000    6,151       24          29      11.0%
 10,000    3,602       21          50       5.2%


## 2. The question: decision, action, cost of a wrong call

**The decision.** Which pages should an editor open first when reviewing titles and meta descriptions, given they can only review a few dozen in a sprint?

**Who acts, and what they do.** A content editor. They open the ranked queue, read the top 20–50 rows with their reason codes, and for each one either rewrite the title/meta, fix an intent mismatch, restructure the snippet, or dismiss it and move on. The output is a review queue, not an automated change. Nothing publishes without a human.

**What a wrong call costs.** Two costs, and they are not symmetric:

- A **false positive** costs editor hours. Someone rewrites a title that was already fine, and the click gap they were chasing was measurement noise. This is the expensive error at the top of a queue, because the top is where the scarce attention goes.
- A **false negative** costs unrealised clicks. A page that genuinely under-captures stays unfixed. Cheaper per instance, but invisible, since nobody notices the queue's omissions.

Because the top of the list is what gets acted on, I optimise for precision at the top and accept lower recall. I also treat low-volume pages as the main false-positive risk: a page with 1,000 impressions and 1 click looks like a large shortfall, and is mostly noise.

**Why data work is needed at all.** The obvious rule already exists in the starter pipeline: flag pages with `ctr < 0.5%` at a decent position. On this data that rule flags most of the eligible population. That is not a queue, that is the inventory. It fails because it uses one flat CTR threshold while the CTR a page *should* earn depends heavily on where it ranks. Expected CTR has to be estimated per position and adjusted for how much volume the estimate rests on. That is modelling work, not an if-statement.

In [2]:
# The plain rule that already ships in scripts/02_baseline_score.py: does it rank anything?
sel = df[(df["impressions_90d"] >= 1000) & (df["avg_position"] > 0)]
flat = sel[(sel["avg_position"] <= 20) & (sel["ctr"] < 0.5)]

print(f"eligible pages (illustrative 1,000-impression floor): {len(sel):,}")
print(f"flagged by 'ctr < 0.5% at position 1-20'            : {len(flat):,} ({len(flat)/len(sel):.1%})")
print("-> a rule that flags most of the inventory is not a queue;")
print("   expected CTR has to vary with position instead of one flat threshold")

eligible pages (illustrative 1,000-impression floor): 13,512
flagged by 'ctr < 0.5% at position 1-20'            : 8,054 (59.6%)
-> a rule that flags most of the inventory is not a queue;
   expected CTR has to vary with position instead of one flat threshold


## 3. Quick look at the data (2-3 real numbers)

Three numbers convinced me this lane is worth the next seven weeks.

**1. Position explains very little of the CTR spread.** Position tier accounts for only about 7% of the variance in per-page CTR (eta-squared ≈ 0.074) **on this slice, where `avg_position` is a 90-day mean. Daily position data from the warehouse may explain more, and re-deriving this is part of the capstone.** Roughly 93% of the spread sits *within* tiers, meaning pages at comparable positions earn very different click rates. That within-tier spread is exactly what this lane ranks. If position explained most of it, there would be nothing left to score. This is the most fragile of my three numbers, and I would rather name that now than defend it later.

**2. But a large share of that spread is noise, not opportunity.** At a 500-impression floor the median page has 5 clicks and half have 5 or fewer. At those counts a single click moves CTR by tenths of a point, so per-page CTR is dominated by small-count variation. This is the central methodological problem of the lane, and it has a principled fix: a volume floor plus shrinking each page's estimate toward its tier's rate in proportion to how thin its evidence is.

**3. The obvious estimator is biased, and I can show it.** Taking the mean of per-page CTR inverts the position ordering: `top_3` comes out *below* `page_1`, which is not credible. Weighting by impressions (total clicks ÷ total impressions) restores a monotonic curve from `top_3` down to `deep`. Mean-of-ratios over-weights tiny-denominator pages, the same small-count problem as point 2 wearing a different hat.

Caveat I am carrying forward: aggregate CTR in this slice runs around 0.31%, while the data dictionary cites roughly 2.78% for positions 1–3 at warehouse scale. The *shape* of the curve is what I rely on here, not the level. Reproducing the curve on the full warehouse release is a required step before I publish any claim about CTR levels.

In [3]:
vis = df[(df["impressions_90d"] >= 500) & (df["avg_position"] > 0)].copy()

# 1. how much of the CTR spread does position tier explain?
within = ((vis["ctr"] - vis.groupby("position_tier")["ctr"].transform("mean")) ** 2).sum()
total = ((vis["ctr"] - vis["ctr"].mean()) ** 2).sum()
print(f"CTR variance explained by position tier (eta^2): {1 - within / total:.3f}")
print(f"-> {within / total:.1%} of the spread is WITHIN tier\n")

# 2. is that spread real signal, or click-count noise?
print(f"median clicks_90d         : {vis['clicks_90d'].median():,.0f}")
for t in (0, 1, 5):
    print(f"  pages with <= {t} clicks : {(vis['clicks_90d'] <= t).sum():>7,} "
          f"({(vis['clicks_90d'] <= t).mean():.1%})")

# 3. mean-of-ratios vs impression-weighted aggregate
tiers = vis.groupby("position_tier").agg(
    pages=("ctr", "size"),
    mean_of_ratios=("ctr", "mean"),
    clicks=("clicks_90d", "sum"),
    impressions=("impressions_90d", "sum"),
)
tiers["aggregate_ctr"] = 100 * tiers["clicks"] / tiers["impressions"]
print("\nexpected CTR by position tier (%):")
print(tiers.reindex(TIER_ORDER)[["pages", "mean_of_ratios", "aggregate_ctr"]].round(3).to_string())

CTR variance explained by position tier (eta^2): 0.074
-> 92.6% of the spread is WITHIN tier

median clicks_90d         : 5
  pages with <= 0 clicks :   2,530 (15.1%)
  pages with <= 1 clicks :   4,544 (27.2%)
  pages with <= 5 clicks :   8,579 (51.3%)

expected CTR by position tier (%):
               pages  mean_of_ratios  aggregate_ctr
position_tier                                      
top_3            458           0.347          0.488
page_1          7064           0.339          0.350
striking        4485           0.267          0.349
page_3_5        4330           0.143          0.155
deep             389           0.043          0.037


## 4. Careful words: what I can and can't claim

**What this work will be able to say (observed / directional / decision-support):**

- *Observed:* in this pseudonymized sample, pages at comparable search positions capture measurably different click rates, and the spread within a position tier is far larger than the spread between tiers.
- *Directional:* pages whose click rate sits well below their position's impression-weighted rate, with enough volume for the estimate to be stable, are reasonable candidates for a title and meta review.
- *Decision-support:* this produces a ranked review queue with reason codes and a confidence band. It orders scarce editor attention. It does not decide anything, and nothing changes without a human reading the page.

**What I will never claim:**

- That rewriting a title **causes** clicks to recover. That needs an experiment or a causal design, and this data supports neither. I can say a page under-captures relative to its position; I cannot say what fixing it would produce.
- That I have measured a Google ranking or algorithm factor. I observe impressions, clicks, and an averaged position, not the mechanism behind any of them.
- That a low CTR means the title or meta description is bad. It is one explanation among several: intent mismatch, a SERP feature absorbing the click, brand recognition, or a competitor's snippet. The queue proposes a *review*, not a diagnosis.
- Anything about AI citations, AI rankings, or AI visibility. `ai_sessions_90d` counts click-throughs only, and is far too sparse here to support claims regardless.

**Limits I will state every time I report a number:** 30,000 pages, 32 pseudonymized clients, one trailing 90-day window, with `avg_position` averaged across that whole window, so a page that moved between positions is being compared against a blended expectation. The starter slice's absolute CTR levels differ from warehouse scale by roughly an order of magnitude, so my published claims will rest on the warehouse curve, not this one. No client names, domains, URLs, or raw queries appear anywhere in my output.

In [4]:
print(f"pseudonymized clients available for grouped splits: {df['client_id'].nunique()}")
print("client_id is used for grouping only, never as a feature")

pseudonymized clients available for grouped splits: 32
client_id is used for grouping only, never as a feature


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.